# Flare Finder Pipeline — TESS Analysis
### Young M Dwarf Flare (YMDF) Framework

This notebook implements the **Flare Finder** pipeline from the YMDF framework for the detection, characterisation, and recovery of stellar flares in TESS photometry time-series data,  Mamonova et al 2025 doi:[10.1051/0004-6361/202554614]


The pipeline follows these steps:
1. **Load** the stellar sample catalogue and raw light curve data
2. **Detrend** the light curve using a Savitzky–Golay filter
3. **Detect** flares using sigma-clipping and multi-threshold criteria
4. **Recover** flare completeness via injection-recovery simulations
5. **Characterise** flares with corrected equivalent durations and energies
6. **Save** the final flare catalogue

This notebook is configured for: TESS 20/60-second cadence photometry data, all parameters are applicable for TESS data.

In [ ]:
## 1. Imports
import pandas
import numpy
import matplotlib.pyplot as plt
from astropy import units, constants
from astropy.table import Table
from typing import Tuple, List, Optional, Any
import seaborn
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pandas.options.mode.chained_assignment = None

from phab.ymdf.flare_finder import (
    findFlares,
    findGaps,
    equivalentDurationModel,
    sigmaClip,
    addPaddingToList,
    detrendSavGol,
    luminosityQuiescent,
    fluxQuiescent,
    findIterativeMedian,
    sampleFlareRecovery,
    characterizeFlares
)

## 2. Stellar Sample Catalogue

Load the pre-compiled stellar sample catalogue (`pandas` DataFrame) containing stellar parameters (ID, radius, effective temperature, distance, etc.) for all targets in the young moving group sample. The catalogue can be stored as a `pandas` pickle file/csv.

In [ ]:
enriched = pandas.read_pickle("./Newsample_reconfirmed.pkl")
print(f"Sample size: {len(enriched)} stars")
print(enriched.head())

## 3. Target Configuration

For TESS photometry the target is identified by name and sector number. Stellar parameters are pulled from the sample catalogue as before.
- Time is converted from TESS barycentric Julian days to seconds
- Flux and error column names (`flux` and `flux_err`)
- Gap detection uses a maximum gap of `0.09 × 24 × 60 × 60` seconds (≈ 7776 s), reflecting the longer TESS orbit-induced gaps
- Minimum continuous observation period is 100 seconds
- `waveRange="photo"` is passed to all YMDF functions`
- Injection-recovery amplitudes and durations are scaled to TESS photometric units


In [ ]:
# convention for the observation files' names
obs    = 1
spur   = 1
prefix = [""]
star   = "2MASS J08440915-7833457"
sector = 66 #actual TESS sector

# Stellar parameters pulled from the sample catalogue
R_star    = enriched.at[star, "star_radius"]
star_teff = enriched.at[star, "teff_lit"]
star_dist = enriched.at[star, "Distance"]

print(f"Target:    {star}")
print(f"Sector:    {sector}")
print(f"R_star:    {R_star} R_sun")
print(f"T_eff:     {star_teff} K")
print(f"Distance:  {star_dist} pc")

## 4. Loading the TESS Light Curve

Load the HST-COS FUV time-series from a CSV file produced by the data reduction pipeline CALCOS. The file contains three columns: time (seconds), flux, and flux error. Flux and error are cast to `float32` to reduce memory usage. The raw time, flux, and error arrays are also extracted as NumPy arrays for use in downstream functions.

In [ ]:
import lightkurve

search_result = lightkurve.search_lightcurve(
    star,
    author="SPOC",
    cadence="fast"
)
print(search_result)

lc      = search_result[0].download()
lcTable = lc.to_pandas()

data12 = pandas.DataFrame(columns=["time", "flux", "fluxError"])
data12["time"]       = lcTable.index * 24 * 60 * 60
data12["flux"]       = lcTable["flux"].values
data12["fluxError"]  = lcTable["flux_err"].values

data12["flux"]       = data12["flux"].astype("float32")
data12["fluxError"]  = data12["fluxError"].astype("float32")

print(f"Light curve loaded: {len(data12)} data points")
print(f"Time range: {data12['time'].min():.1f} – {data12['time'].max():.1f} s")


### Alternative: Loading from Local FITS Cache

If the TESS light curve has already been downloaded and is stored in the local `lightkurve` cache or on the cluster storage, it can be loaded directly from the FITS file using `fitsToPandas` from the `phab` utilities, bypassing the MAST query entirely. This is the preferred approach when working on the UiO cluster (Kant/Hypatia) where repeated MAST queries are slow or network access is restricted.

Note that TESS flux values are stored in electrons per second in the FITS files and must be converted to normalised flux units using `tess_electrons_to_flux` before passing to the YMDF pipeline. Time is stored in TESS barycentric Julian days and is converted to seconds here for consistency with the HST-COS pipeline.


In [ ]:
from phab.utils.databases.lightcurves import fitsToPandas

pathd = (
    "./.lightkurve/cache/mastDownload/TESS/"
    "tess2020186164531-s0027-0000000441420236-0189-a_fast/"
    "tess2020186164531-s0027-0000000441420236-0189-a_fast-lc.fits"
)

data = fitsToPandas(pathd)
print(data.head())

# Convert time from TESS BJD days to seconds
timeSeries      = data["time"].values * 24. * 60. * 60.
fluxSeries      = data["flux"].values
fluxErrorSeries = data["fluxError"].values


data["time"]       = timeSeries
data["flux"]       = fluxSeries
data["fluxError"]  = fluxErrorSeries

data12 = data.copy()
print(f"Light curve loaded: {len(data12)} data points")
print(f"Time range: {data12['time'].min():.1f} – {data12['time'].max():.1f} s")


In [ ]:
# Minimal illustration of the stitching logic inside run_flare_pipeline:

all_flares = pandas.DataFrame()   # empty container

for sector, author, cadence in plan:
    data   = download_sector(star_name, sector, author, cadence)
    if data is None:
        continue
    result = process_sector(data, sector, author, cadence,
                            star_name, luminosity)
    if result is not None and not result.empty:
        all_flares = pandas.concat([all_flares, result], ignore_index=True)

# all_flares now contains every flare from every sector in one DataFrame.
# Save to disk:
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_base = f"{OUTPUT_DIR}/{star_name}_flares_TESS_short"
all_flares.to_pickle(f"{out_base}.pkl")
all_flares.to_csv(f"{out_base}.csv")
print(f"Total flares across all sectors: {len(all_flares)}")
print(f"Sectors with flares: {sorted(all_flares['sector'].unique())}")

## 5. Gap Detection and Detrending

HST-COS observations are interrupted by Earth occultations, producing regular gaps in the time series. The `findGaps` function identifies these gaps using a maximum allowed gap of 30 seconds and a minimum continuous observation period of 10 seconds. The `detrendSavGol` function then fits and removes the slowly varying quiescent stellar continuum within each continuous segment using a Savitzky–Golay filter, producing a detrended light curve suitable for flare detection.

In [ ]:
gaps = findGaps(
    data12,
    maximumGap=0.09 * 24. * 60. * 60.,
    minimumObservationPeriod=100
)
print(f"Number of continuous segments: {len(gaps)}")

data1 = detrendSavGol(
    data12,
    gaps,
    padding=3,
    waveRange="photo",
    windowLength=0
)
data1 = data1.dropna(subset=["time"])

## 6. Flare Detection

Flares are identified in the detrended light curve using a multi-threshold sigma-clipping algorithm implemented in `findFlares`.  A candidate flare requires the detrended flux to exceed the local noise level by at least `n1=3` sigma, and to simultaneously exceed `n2=2` times the combined sigma and detrended flux error. At least `n3=3` consecutive points must satisfy these criteria for an event to be flagged as a flare. A minimum separation of 3 data points is enforced between distinct flare events. The function returns the updated detrended light curve `data1` and a flare table `fla1` containing the detected event: start and stop in indices and actual time, recovered equivalent duration, recovered amplitude, duration and the total valid data points.


In [ ]:
data1, fla1 = findFlares(
    data12,
    n1=3,
    n2=2,
    n3=3,
    minSep=3,
    padding=3,
    maximumGap=0.09 * 24. * 60. * 60.,
    minimumObservationPeriod=100,
    waveRange="photo"
)

print(f"Flares detected: {len(fla1)}")
print(fla1)

## 8. Injection-Recovery Simulations

To characterise the completeness of the flare detection as a function of flare amplitude and duration, synthetic flares are injected into the observed light curve and the recovery rate is measured. The `sampleFlareRecovery` function generates 3500 synthetic flares drawn from a uniform distribution in amplitude (0.01–200 normalised flux units) and duration (10–10000 s), injects them one at a time into the detrended light curve, and attempts to recover each using the same detection thresholds as in the previous step. The injection rate is set to one synthetic flare per observation second (`fakefreq=1./24/60/60`), with a maximum of one injected flare per continuous segment to avoid confusion between overlapping events.

The output consists of two tables:
- `fla_rec` — the recovered flares
- `fla_inj` — the properties of all injected synthetic flares

In [ ]:
fla_rec, fla_inj = sampleFlareRecovery(
    data1,
    fla1,
    iterations=1500,
    n1=3,
    n2=2,
    n3=3,
    minSep=3,
    padding=3,
    maximumGap=0.09 * 24. * 60. * 60.,
    minimumObservationPeriod=100,
    ampl=[1e-4, 5.],
    dur=[0.0005 * 24 * 60 * 60, 0.12 * 24 * 60 * 60],
    fakefreq=1. / 24 / 60 / 60,
    maxFlaresPerGap=1
)

print(f"Total injected flares:  {len(fla_inj)}")
print(f"Total recovered flares: {len(fla_rec)}")


## 9. Flare Characterisation

The `characterizeFlares` function uses the injection-recovery results to compute completeness-corrected equivalent durations (`ed_corr`) for each detected flare. 2D recovery heatmaps in amplitude–duration space for probability and ED ratio is generated to visualise the detection completeness across the parameter space sampled by the injection-recovery simulations. The corrected equivalent durations can subsequently be converted to flare energies using the quiescent stellar luminosity.

In [ ]:
fla3 = characterizeFlares(
    fla_rec,
    fla_inj,
    ampl=[0.01, 200.],
    dur=[10., 10000.],
    plot_heatmap=True
)

print(f"Characterised flares: {len(fla3)}")
print(fla3)

## 10. Saving Results

The final characterised flare catalogue is saved in both CSV and pickle.

In [ ]:
fla3.to_csv(f"./new_sample_flares/{star}_{sector}_flares_recovered.csv")
fla3.to_pickle(f"./new_sample_flares/{star}_{sector}_flares_recovered.pkl")

print(f"Saved: ./new_sample_flares/{star}_{sector}_flares_recovered.csv")
print(f"Saved: ./new_sample_flares/{star}_{sector}_flares_recovered.pkl")